<a href="https://colab.research.google.com/github/YoussefAli07/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review



This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
!git clone https://github.com/YoussefAli07/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [23]:
import pandas as pd
df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

## 1. My rule and its reason codes

### Signal check 1: CTR gap vs. trend_pct

**Hypothesis:** pages with a large negative CTR gap (underperforming their position's
average CTR, using the ML-02 threshold of ≤ -0.5pp) should show lower trend_pct than
pages that are not underperforming.

**Method:** grouped pages into two buckets (ctr_gap ≤ -0.5 vs. > -0.5) and compared
mean trend_pct per group, with number of rows printed to check group sizes.

**Result:** non-underperforming pages (n=21,006) average -0.51 trend_pct;
underperforming pages (n=5,606) average -20.81 trend_pct.

**Verdict:** CONFIRMED. underperforming pages show substantially lower trend_pct
(a ~20 point gap), and both groups have large sample sizes.


### Signal check 2: staleness (freshness_tier) vs. trend_pct

**Hypothesis:** pages with more days since their last update (staler pages)
should show lower trend_pct than fresher pages.

**Method:** grouped pages by freshness_tier (0-30, 31-90, 91-180, 181+) and
compared trend_pct for each group. Mean was checked first, but the two smallest
tiers (31-90, n=167; 181+, n=133) had extreme outliers (max values up to 1080)
skewing their means. So I used median instead as a summary statistic.

**Result:** median trend_pct by freshness_tier (logical order): 0-30 = -33.3, 31-90 = -38.5, 91-180 = -34.4, 181+ = -44.5.

**Verdict:** CONFIRMED. median trend_pct declines from [-33.3] to
[-44.5] as staleness increases. A minor tick at 91-180
(n=8,858, a large stable group) does not meaningfully undermine the overall
declining trend.

**Rebuilding the CTR gap column that I had in ML-02 and show the average trend_pct for underperforming and not-underperforming pages.**

In [24]:
df['position_avg_ctr'] = df.groupby('position_tier')['ctr'].transform('mean')
df['ctr_gap'] = df['ctr'] - df['position_avg_ctr']
df.groupby('position_tier')['ctr_gap'].mean()
underperforming = df['ctr_gap'] <= -0.5
df.groupby(underperforming)['trend_pct'].agg(['mean', 'count'])

,mean,count
ctr_gap,,
False,-0.510240,21006
True,-20.807367,5606


**Signal Check 2 Group by result**

In [25]:
df['days_since_last_update'].isnull().sum()
df.groupby("freshness_tier")["trend_pct"].agg(["median", "count"])

# df[df['freshness_tier'] == '181+']['trend_pct'].describe()
# df[df['freshness_tier'] == '31-90']['trend_pct'].describe()

# df['impressions_90d'].describe()


,median,count
freshness_tier,,
0-30,-33.3,17454
181+,-44.5,133
31-90,-38.5,167
91-180,-34.4,8858


### My rule (in plain words)

A page is worth reviewing for a title/meta rewrite if it is:
- **Visible:** impressions_90d ≥ 81 (25th percentile. A light filter, since
  staleness already narrows the number of rows sharply)
- **Stale:** freshness_tier == '181+' (the one tier clearly  worse than the rest, middle tiers were too close together to split comfortably)
- **Underperforming:** ctr_gap ≤ -0.5 (position-adjusted CTR threshold from ML-02)

### **Reason codes :**

My original plan was to split reason codes into severity tiers, so something like:
("severely underperforming" vs. "underperforming") based on ctr_gap.
However, investigation into the flagged rows showed all 9 pages share an
identical ctr_gap (-0.652467) because they all belong to the same position_tier
(page_1) and all share ctr = 0.0.

After checking clicks_90d confirmed this isn't missing-data : each of these
9 pages has non-zero impressions_90d (ranging from 81 to 429) but exactly
0 clicks_90d. In other words, these should be pages that were genuinely shown to
searchers but received not a single click in 90 days. And that's the most extreme form
of underperformance this rule can catch.

Since all 9 flagged pages share this identical pattern, splitting them into severity tiers wouldn't mean something. Instead, one honest
reason code is used: `zero_clicks_visible_but_stale`.

### **Action Label :**
As those are underperforming pages that potentially would need a title/meta rewrite, the action label is direct, plain, and tells who reads it that this page needs immediate review. 'Needs review — potentially a title/meta rewrite'


In [26]:
visible = df['impressions_90d'] >= 81
stale = df['freshness_tier'] == '181+'
underperforming = df['ctr_gap'] <= -0.5

flagged = df[visible & stale & underperforming].copy()
flagged.shape

flagged['score'] = flagged['ctr_gap'].abs() * flagged['impressions_90d']

flagged = flagged.sort_values('score', ascending=False).reset_index(drop=True)

In [27]:
flagged[['content_id', 'ctr_gap', 'impressions_90d', 'score']].sort_values('ctr_gap')
flagged['reason_code'] = 'zero_clicks_visible_but_stale'

In [28]:
flagged['action_label'] = 'Needs review — potentially a title/meta rewrite'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [33]:
import os
os.makedirs('flyrank-ml-internship-starter/work/outputs', exist_ok=True)

flagged.to_csv('flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv', index=False)


In [30]:
flagged[['content_id', 'score', 'impressions_90d', 'action_label']]



,content_id,score,impressions_90d,action_label
0,content_fd16e3475c29,279.908156,429,Needs review — potentially a title/meta rewrite
1,content_ea41fe5cf292,172.903640,265,Needs review — potentially a title/meta rewrite
2,content_958a46db26bd,129.188380,198,Needs review — potentially a title/meta rewrite
3,content_02b0d6e30129,114.834115,176,Needs review — potentially a title/meta rewrite
4,content_f488400fca67,101.132318,155,Needs review — potentially a title/meta rewrite
5,content_ab27c30d81f4,67.204056,103,Needs review — potentially a title/meta rewrite
6,content_07ce98c6085a,55.459658,85,Needs review — potentially a title/meta rewrite
7,content_30eb41dff556,54.807191,84,Needs review — potentially a title/meta rewrite
8,content_460b11dcac6a,52.849792,81,Needs review — potentially a title/meta rewrite


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


## 3. Top-9 review

(Only 9 pages passed all three gates: visible, stale, underperforming. So all 9
are reviewed below instead of a top-10.)

1. **content_fd16e3475c29** (score 279.9, 429 impressions)

 Needs review/rewrite.
   Highest-traffic page in the flagged set with zero clicks in 90 days. Would be
   wrong if this page's target query no longer matches its content's intent,
   since a title/meta fix can't repair an intent mismatch.

2. **content_ea41fe5cf292** (score 172.9, 265 impressions)

 Needs review/rewrite.
   Strong visibility with no conversion to clicks. Would be wrong if impressions
   are inflated by bots rather than real searchers.

3. **content_958a46db26bd** (score 129.2, 198 impressions)

Needs review/rewrite.
   Solid impressions , zero clicks, stale content. Would be wrong if the
   page's ranking position dropped so far post-snapshot that impressions no
   longer reflect current visibility.

4. **content_02b0d6e30129** (score 114.8, 176 impressions)

 Needs review/rewrite.
   Moderate visibility with zero engagement. Would be wrong if this content_type
   has systematically unreliable click tracking.

5. **content_f488400fca67** (score 101.1, 155 impressions)

 Needs review/rewrite.
   Meaningful impressions, no clicks, stale. Would be wrong if the SERP snippet
   (not the title/meta itself) is what's suppressing clicks.

6. **content_ab27c30d81f4** (score 67.2, 103 impressions)

 Needs review/rewrite.
   Lower but still has traffic with zero clicks. Would be wrong if search_volume
   for its target keyword has genuinely declined, making the opportunity smaller
   than the score suggests.

7. **content_07ce98c6085a** (score 55.5, 85 impressions)

 Needs review/rewrite.
   Smaller audience, same zero-click pattern. Would be wrong if this page's low
   position (not the title/meta) is the real reason it's not getting clicks.

8. **content_30eb41dff556** (score 54.8, 84 impressions)

Needs review/rewrite.
   Similar profile to #7, near the bottom of the flagged set by traffic. Would
   be wrong if the page is a near-duplicate of a better-performing page and
   traffic is being transfered elsewhere.

9. **content_460b11dcac6a** (score 52.8, 81 impressions)

 Needs review/rewrite.
   Smallest audience among the flagged pages (just above the visibility floor of
   81). Would be wrong if this page's impression count is too close to the
   visibility threshold to give it a meaningful opportunity.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


**Weak Picks :**


**content_460b11dcac6a** (score 52.8, 81 impressions) is the shakiest pick in
this flagged set. It has the lowest impressions_90d among all 9 rows. Sitting
right at the visibility threshold of 81, the minimum allowed by the rule. This
means it qualifies only barely.

**No leakage found :**


The rule uses only impressions_90d, freshness_tier, and ctr_gap. None of which are derived from trend_pct/trend_direction, so no label leakage is present."

In [31]:
used_columns = ['impressions_90d', 'freshness_tier', 'ctr_gap']
leakage_cols = ['trend_pct', 'trend_direction', 'is_declining_label']
print(any(c in used_columns for c in leakage_cols))

False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.